# 03 — Matrix Multiplication

**Ladder:** Naive (1 output / pid) → blocked tiles → **K-tiling** → L2-friendly **grouped** program order → Triton.

**Study tip.** Work the teaching sizes on the boards ($6\times3@3\times8$, then $6\times8@8\times4$ with $B_M,B_N,B_K=3,2,4$) before large benches. Grouped pid ≠ Group GEMM (folder 08).

Official: [Matrix Multiplication](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html)

Handwritten pack: [`docs/visual-guide.pdf`](docs/visual-guide.pdf) · figures: [`docs/images/`](docs/images/)

---

# Part I — Mathematics

## 0. Definition, FLOPs, and arithmetic intensity

Let $A\in\mathbb{R}^{M\times K}$ and $B\in\mathbb{R}^{K\times N}$. The product
$C=AB\in\mathbb{R}^{M\times N}$ is

$$
C_{mn}
=
\sum_{k=0}^{K-1}A_{mk}B_{kn}
=
\langle A_{m,:},\,B_{:,n}\rangle,
\qquad m\in[M],\ n\in[N].
\tag{0.1}
$$

**Proposition 1 (bilinearity).** The map $(A,B)\mapsto AB$ is linear in $A$ for
fixed $B$ and linear in $B$ for fixed $A$:

$$
(\alpha A+\alpha'A')B=\alpha AB+\alpha'A'B,
\qquad
A(\beta B+\beta'B')=\beta AB+\beta'AB'.
\tag{0.2}
$$

*Proof.* Expand (0.1) and use distributivity of scalar multiplication in
$\mathbb{R}$. $\square$

Bilinearity is why block scaling, masking, and grouping are all algebraically
safe: they only reorganize sums and scalar factors.

**FLOP count.** Each $(m,n,k)$ contributes one multiply and one add. Hence

$$
F=2MNK
\qquad
\text{(counting a fused multiply–add as 2 FLOPs)}.
\tag{0.3}
$$

**Memory traffic (no reuse).** A naive implementation reads each $A_{mk}$ once
per output column $n$ and each $B_{kn}$ once per output row $m$:

$$
Q_{\mathrm{naive}}
=
\underbrace{MNK}_{\text{loads of }A}
+
\underbrace{MNK}_{\text{loads of }B}
+
\underbrace{MN}_{\text{stores of }C}
\ \text{elements}.
\tag{0.4}
$$

**Arithmetic intensity.** With element size $b$,

$$
\mathrm{AI}_{\mathrm{naive}}
=
\frac{2MNK}{(2MNK+MN)b}
\;\xrightarrow[K\to\infty]{}\;
\frac{1}{b}
\ \text{FLOP/byte}.
\tag{0.5}
$$

For `float16` ($b=2$), $\mathrm{AI}\to0.5$ FLOP/byte, whereas tensor cores need
tens to hundreds of FLOP/byte. This gap is the entire reason for **blocking**:
reuse operands from registers/SRAM so DRAM traffic grows sublinearly in $K$.

**Blocked intensity (preview).** A tile of size $B_M\times B_N$ computed with
$K$-panels of width $B_K$ loads

$$
B_MB_NB_K \text{ MACs}
\quad\text{while reading}\quad
(B_M+B_N)B_K \text{ elements},
$$

so per $K$-panel

$$
\mathrm{AI}_{\mathrm{tile}}
=
\frac{2B_MB_NB_K}{(B_M+B_N)B_Kb}
=
\frac{2B_MB_N}{(B_M+B_N)b}.
\tag{0.6}
$$

For $B_M=B_N=128$, $b=2$: $\mathrm{AI}=128/2=64$ FLOP/byte — a $128\times$
improvement over (0.5). This single formula explains every tiling parameter in
the notebook.

### 0.1 Worked example

$$
A=
\begin{pmatrix}1&2\\3&4\end{pmatrix},
\qquad
B=
\begin{pmatrix}5&6\\7&8\end{pmatrix}.
$$

Then

$$
C=
\begin{pmatrix}
1\cdot5+2\cdot7 & 1\cdot6+2\cdot8\\
3\cdot5+4\cdot7 & 3\cdot6+4\cdot8
\end{pmatrix}
=
\begin{pmatrix}19&22\\43&50\end{pmatrix}.
$$

$F=2\cdot2\cdot2\cdot2=16$ FLOPs, matching four dot products of length two.

### 0.2 Running examples (from the notes)

| Stage | Product | Blocks |
|-------|---------|--------|
| Naive / early blocked | $A_{6\times3}B_{3\times8}\to C_{6\times8}$ | — / $B_M=3,\,B_N=4$ |
| K-tiled | $A_{6\times8}B_{8\times4}\to C_{6\times4}$ | $B_M=3,\,B_N=2,\,B_K=4$ |

For the second: $F=2\cdot6\cdot4\cdot8=384$ FLOPs, four output tiles, two
$K$-panels per tile.

### Hardware mental model (before kernels)

![Grid → Thread Block → Warp → Threads](docs/images/grid-block-warp-thread.jpeg)

![CUDA hierarchy: grid, blocks, warps of 32, SIMT](docs/images/cuda-execution-hierarchy.png)

Triton **programs** correspond to CUDA blocks. A matmul launch is a 1-D grid;
each program owns either one $C_{mn}$ (naive) or one $B_M\times B_N$ tile
(blocked / grouped).

# Part II — Proof

**Theorem (blocked product).** With tiles
$C_{p_m,p_n}=C[p_mB_M:(p_m+1)B_M,\,p_nB_N:(p_n+1)B_N]$ and $K$-panels
$A_t=A[:,\,tB_K:(t+1)B_K]$, $B_t=B[tB_K:(t+1)B_K,\,:]$,
the loop

$$
\mathrm{Acc}^{(0)}=0,
\qquad
\mathrm{Acc}^{(t+1)}=\mathrm{Acc}^{(t)}+A_tB_t
$$

satisfies $\mathrm{Acc}^{(G_K)}=C_{p_m,p_n}$.

*Proof by induction.* The invariant before step $t$ is
$\mathrm{Acc}^{(t)}=\sum_{s<t}A_sB_s$. The base is the empty sum; the step adds
the $t$-th term. At $t=G_K$ the sum is the full block product. $\square$

**Corollary (tile coverage).** The tile map
$\mathrm{pid}\mapsto(p_m,p_n)$ is a bijection onto
$[G_M]\times[G_N]$, so the union of tiles is $C$.

*Proof.* Quotient–remainder with divisor $G_N$. $\square$

**Corollary (grouped swizzle).** The grouped schedule is a permutation of the
tile grid; it changes which program computes a tile, never the value of $C$.

*Proof.* The grouped map partitions tile rows into groups and enumerates each
group in column-major order, which is a bijection. Each tile formula is
unchanged. $\square$

# Part III — Step-by-step proof

1. **Tile the output.** Choose $B_M,B_N$; compute $G_M=\lceil M/B_M\rceil$,
   $G_N=\lceil N/B_N\rceil$ and $T=G_MG_N$.
2. **Map pid to tile.** $p_m=\lfloor\mathrm{pid}/G_N\rfloor$,
   $p_n=\mathrm{pid}\bmod G_N$.
3. **Build offsets.** $o_m=p_mB_M+\{0,\dots,B_M-1\}$,
   $o_n=p_nB_N+\{0,\dots,B_N-1\}$.
4. **Initialise accumulator.** $\mathrm{Acc}=0\in\mathbb{R}^{B_M\times B_N}$
   in FP32.
5. **Loop over K-panels.** For $t=0,\dots,G_K-1$, load the masked panels
   $A_t\in\mathbb{R}^{B_M\times B_K}$ and $B_t\in\mathbb{R}^{B_K\times B_N}$.
6. **Accumulate.** $\mathrm{Acc}\leftarrow\mathrm{Acc}+A_tB_t$; this is exactly
   the induction step above.
7. **Advance pointers.** Add $B_Ks_{ak}$ and $B_Ks_{bk}$.
8. **Store.** Cast $\mathrm{Acc}$ to the output dtype and store with masks.
9. **Reorder for L2 (optional).** Replace step 2 by the grouped swizzle; the
   induction and the result are unchanged.

# Part IV — Coding

The coding part follows the three-rung ladder used in the PDFs: **Method 01 — naive**, **Method 02 — kernel-based**, and **Method 03 — fused kernel-based**. Each method is runnable and is checked against the previous one.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pytest
import torch
import triton
import triton.language as tl

def _active_device():
    """Pick Triton active device, else CUDA, else CPU (study helper; no common/ package)."""
    try:
        import triton

        try:
            return triton.runtime.driver.active.get_active_torch_device()
        except Exception:
            pass
    except Exception:
        pass
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


DEVICE = _active_device()


def require_gpu() -> None:
    if not torch.cuda.is_available() and getattr(DEVICE, "type", None) != "cuda":
        pytest.skip(f"GPU not available (active device: {DEVICE})")



### Method 01 — Naive (Python)

![Naive: one program per C[m,n]; m=i//N, n=i%N](docs/images/matmul-naive-threads.png)

![Notes — naive overview](docs/images/notes-01-naive-overview.png)

Enumerate output entries by a flat program id
$\mathrm{pid}\in\{0,\dots,MN-1\}$:

$$
m=\left\lfloor\frac{\mathrm{pid}}{N}\right\rfloor,
\qquad
n=\mathrm{pid}\bmod N.
\tag{1.1}
$$

**Lemma 1 (bijection).** The map $\mathrm{pid}\mapsto(m,n)$ is a bijection onto
$[M]\times[N]$.

*Proof.* The quotient–remainder theorem gives unique $m,n$ with
$\mathrm{pid}=mN+n$ and $0\le n<N$. $\square$

Each program then computes

$$
C_{mn}=\sum_{k=0}^{K-1}A_{mk}B_{kn}.
\tag{1.2}
$$

**Concrete check.** For $M=6$, $N=8$ there are $48$ programs; pid $19$ writes
$C[2,3]$ because $19=2\cdot8+3$.

### 1.1 Access pattern and cost

The inner loop reads

$$
A_{m,k}\ \text{at address } \&A+mK+k
\quad\text{(contiguous in }k\text{)},
\qquad
B_{k,n}\ \text{at address } \&B+kN+n
\quad\text{(stride }N\text{ in }k\text{)}.
\tag{1.3}
$$

Thus each program performs one contiguous row read of length $K$ and one
strided column read of length $K$. Across programs:

- row $A_{m,:}$ is read once per output column → $N$ times;
- column $B_{:,n}$ is read once per output row → $M$ times.

This reproduces the traffic count (0.4):

$$
Q_{\mathrm{naive}}
=
MNK+MNK+MN
=
(2K+1)MN
\ \text{elements}.
\tag{1.4}
$$

**Latency.** Each program executes a serial reduction of length $K$ with a loop
carried dependency on `acc`. There is little instruction-level parallelism, and
the strided $B$ reads cannot be coalesced. The kernel is correct but both
bandwidth- and latency-limited.

**Warp view.** Consecutive pids differ in $n$ (same $m$ for a run of $N$ pids).
Within one warp, the 32 lanes read 32 different columns of $B$ at the same $k$;
those addresses are contiguous in $n$! So the *cross-program* pattern is
coalesced even though each program's own access is strided. The inefficiency is
therefore mainly **redundant traffic**, not uncoalesced access. This distinction
motivates blocking: keep the coalescing, remove the reloads.


In [ ]:
def naive_matmul_psuedo(A, B):
    """
    Matmul is simple! We have two matricies:
    (See markdown cells above for the math.)
    """

    M, K = A.shape
    K, N = B.shape
    C = torch.zeros(M, N, device=A.device, dtype=A.dtype)

    def dot(v1, v2):
        accum = 0
        for p, q in zip(v1, v2):
            accum += p * q
        return accum

    num_pids = list(range(M * N))  # [0, 1, 2, ... m*n]
    for i in num_pids:
        ### Get our m,n index from our position in the threads ###
        ### Each row of our output matrix is N long, so we can get ###
        ### which row we are on with a simple division operation ###
        m = i // N

        ### And we can get which column we are on with a simple modulo operation ###
        n = i % N

        C[m, n] = dot(A[m, :], B[:, n])

    return C


### Method 02 — Kernel-based (direct Triton)

This is the literal kernel translation of the naive algorithm: one program per
output element $C_{mn}$, a serial loop over $k$, and scalar loads/stores. It is
correct but does not reuse either operand — the baseline against which blocking
is measured.


In [ ]:
@triton.jit
def naive_matmul_kernel(
    A_ptr,
    B_ptr,
    C_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
):
    # Program ID — each "program" computes one element of C
    pid = tl.program_id(axis=0)  # 0..M*N - 1

    # Compute (m, n) coordinates of this output element
    m = pid // N
    n = pid % N

    # If we're out of bounds, exit
    if (m >= M) or (n >= N):
        return

    # Compute the dot product between A[m, :] and B[:, n]
    acc = 0.0
    for k in range(0, K):
        a_val = tl.load(A_ptr + m * stride_am + k * stride_ak)
        b_val = tl.load(B_ptr + k * stride_bk + n * stride_bn)
        acc += a_val * b_val

    # Write result to output
    tl.store(C_ptr + m * stride_cm + n * stride_cn, acc)


def naive_matmul(A, B):
    M, K = A.shape
    K2, N = B.shape
    assert K == K2

    C = torch.zeros((M, N), device=A.device, dtype=A.dtype)

    grid = (M * N,)

    naive_matmul_kernel[grid](
        A,
        B,
        C,
        M,
        N,
        K,
        A.stride(0),
        A.stride(1),
        B.stride(0),
        B.stride(1),
        C.stride(0),
        C.stride(1),
    )
    return C


### Method 03 — Fused kernel-based (blocked)

![Blocked tile $B_M=3$, $B_N=2$](docs/images/notes-05-blocked-tile.png)

![K-tiling: $A_{3\times4}B_{4\times2}\to C_{3\times2}$](docs/images/notes-08-k-tiling.png)

![Two $B_K$ panels accumulate into one C-tile](docs/images/notes-13-k-accumulation.png)

Choose tile sizes $B_M,B_N$ and, along the reduction, $B_K$. For the teaching
case assume divisibility $B_M\mid M$, $B_N\mid N$, $B_K\mid K$. Define

$$
G_M=\frac{M}{B_M},
\qquad
G_N=\frac{N}{B_N},
\qquad
G_K=\frac{K}{B_K}.
\tag{2.1}
$$

Program $\mathrm{pid}\in\{0,\dots,G_MG_N-1\}$ owns the output tile

$$
(p_m,p_n)
=
\left(
\left\lfloor\frac{\mathrm{pid}}{G_N}\right\rfloor,
\ \mathrm{pid}\bmod G_N
\right),
\tag{2.2}
$$

and writes

$$
C\big[p_mB_M:(p_m+1)B_M,\ p_nB_N:(p_n+1)B_N\big]
=
A\big[p_mB_M:(p_m+1)B_M,\ :\big]\,
B\big[:,\ p_nB_N:(p_n+1)B_N\big].
\tag{2.3}
$$

**Lemma 2 (tile map is a bijection).** (2.2) is a bijection
$[G_MG_N]\to[G_M]\times[G_N]$.

*Proof.* Same quotient–remainder argument with divisor $G_N$. $\square$

**Corollary 3 (union of tiles).** The tiles in (2.3) are disjoint and cover
$C$. Hence computing every tile yields the unique $C=AB$.

*Proof.* Lemma 2 enumerates all tile coordinates; the index ranges partition
$[M]$ and $[N]$. $\square$

### 2.1 K-tiling and the loop invariant

Do not take the full $K$ strip at once. Split the reduction into panels:

$$
C_{\mathrm{tile}}
=
\sum_{t=0}^{G_K-1}
A^{(t)}_{\mathrm{tile}}\,B^{(t)}_{\mathrm{tile}},
\qquad
A^{(t)}=A[:,\,tB_K:(t+1)B_K],
\quad
B^{(t)}=B[tB_K:(t+1)B_K,\,:].
\tag{2.4}
$$

**Algorithm (per tile).**

$$
\mathrm{Acc}^{(0)}=0\in\mathbb{R}^{B_M\times B_N},
\qquad
\mathrm{Acc}^{(t+1)}=\mathrm{Acc}^{(t)}+A^{(t)}_{\mathrm{tile}}B^{(t)}_{\mathrm{tile}}.
\tag{2.5}
$$

**Invariant.** Before iteration $t$,

$$
\mathrm{Acc}^{(t)}
=
\sum_{s=0}^{t-1}A^{(s)}_{\mathrm{tile}}B^{(s)}_{\mathrm{tile}}.
\tag{2.6}
$$

*Proof by induction.* Base: empty sum. Step: add the $t$-th term. At $t=G_K$ the
invariant gives (2.4), hence $C_{\mathrm{tile}}$. $\square$

**Accumulator precision.** $\mathrm{Acc}$ is `float32` even when $A,B$ are
`float16`. Each `tl.dot` may use tensor cores internally, but the accumulation
across panels is FP32, which keeps the error close to the FP32 rounding level
instead of accumulating in half precision.

### 2.2 Work, traffic, and intensity per tile

Per tile, per $K$-panel:

$$
\text{FLOPs}=2B_MB_NB_K,
\qquad
\text{loads}=(B_M+B_N)B_K\ \text{elements},
\qquad
\text{store}=B_MB_N.
\tag{2.7}
$$

Summing over $G_K$ panels and $G_MG_N$ tiles:

$$
F=2MNK,
\qquad
Q_{\mathrm{blocked}}
=
G_MG_N\big[(B_M+B_N)K+B_MB_N\big]
\ \text{elements}.
\tag{2.8}
$$

The first term is the panel traffic; the second is the output. The ratio
$Q_{\mathrm{blocked}}/Q_{\mathrm{naive}}$ shrinks like $1/B_M+1/B_N$ relative to
$K$, which is the quantitative statement of reuse.

### 2.3 Teaching walk ($M=6$, $N=8$, $K=3$, $B_M=3$, $B_N=4$)

$G_M=G_N=2$, so four programs:

| pid | $(p_m,p_n)$ | Writes |
|-----|-------------|--------|
| 0 | $(0,0)$ | $C[0:3,0:4]=A[0:3,:]B[:,0:4]$ |
| 1 | $(0,1)$ | $C[0:3,4:8]$ |
| 2 | $(1,0)$ | $C[3:6,0:4]$ |
| 3 | $(1,1)$ | $C[3:6,4:8]$ |

**K-tiled example.** $A_{6\times8}B_{8\times4}$, $B_M=3$, $B_N=2$, $B_K=4$:
$G_M=2$, $G_N=2$, $G_K=2$, so four programs, each doing two `tl.dot` steps:

$$
\mathrm{Acc}
=
A[:,0:4]B[0:4,:]
+
A[:,4:8]B[4:8,:].
$$

### 2.4 Row-major tile order and its weakness

Row-major assigns consecutive pids across a row of tiles:
$(0,0),(0,1),\dots,(0,G_N-1),(1,0),\dots$. While a tile row is processed, the
$A$ panel $A[p_mB_M:(p_m+1)B_M,:]$ is reused $G_N$ times and can stay in L2.
But when moving to the next tile row, the whole $B$ matrix is streamed again.
Thus

$$
\text{A panel streams}\approx G_M,
\qquad
\text{B panel streams}\approx G_MG_N
\tag{2.9}
$$

(up to cache capacity). If $B$ is large, the second term dominates. Grouped
ordering in §3 fixes exactly this asymmetry.


In [ ]:
def blocked_matmul_pseudo(A, B, BLOCK_SIZE_M, BLOCK_SIZE_N):
    """
    We typically do GPU compute in blocks. Instead of having every
    (See markdown cells above for the math.)
    """

    M, K = A.shape
    K, N = B.shape
    C = torch.zeros(M, N, device=A.device, dtype=A.dtype)

    assert M % BLOCK_SIZE_M == 0, "Assuming M divisible by BLOCK_SIZE_M for simplicity"
    assert N % BLOCK_SIZE_N == 0, "Assuming N divisible by BLOCK_SIZE_N for simplicity"

    ### Get how many rows and cols we have at the Block level ###
    num_block_rows = M // BLOCK_SIZE_M
    num_block_cols = N // BLOCK_SIZE_N

    ### Now each pid refers to a block of our output, not just a single output
    num_pids = list(range(num_block_rows * num_block_cols))
    for i in num_pids:
        ### This gives us the top left index of every block
        block_m = i // num_block_cols  # What row are we on?
        block_n = i % num_block_cols  # What col are we on?

        ### But we need to convert block_m, block_n to actual indexes in our data
        m = block_m * BLOCK_SIZE_M
        n = block_n * BLOCK_SIZE_N

        ### Grab the corresponding block of rows and cols from A and B
        block_A = A[m : m + BLOCK_SIZE_M, :]
        block_B = B[:, n : n + BLOCK_SIZE_N]

        ### Now its basically a tiny matmul between block_A and block_B!
        C[m : m + BLOCK_SIZE_M, n : n + BLOCK_SIZE_N] = naive_matmul_psuedo(block_A, block_B)

    return C


#### L2-friendly grouped ordering

![Row-major vs grouped A/B load counts](docs/images/notes-19-group-load-cost.png)

![Grouped pid: column-major inside each group](docs/images/notes-20-grouped-pid.png)

Fix a group height $G$ (Triton: `GROUP_SIZE_M`). Let

$$
\mathrm{num\_pid\_in\_group}=G\,G_N.
\tag{3.1}
$$

For program $\mathrm{pid}$:

$$
\begin{aligned}
\mathrm{group\_id} &= \left\lfloor \frac{\mathrm{pid}}{G\,G_N}\right\rfloor,\\
\mathrm{first\_pid\_m} &= \mathrm{group\_id}\cdot G,\\
g_m &= \min(G_M-\mathrm{first\_pid\_m},\,G),\\
w &= \mathrm{pid}\bmod(G\,G_N),\\
p_m &= \mathrm{first\_pid\_m}+(w\bmod g_m),\\
p_n &= \left\lfloor \frac{w}{g_m}\right\rfloor .
\end{aligned}
\tag{3.2}
$$

**Interpretation.** Within a group, pids advance in **column-major** order over a
$g_m\times G_N$ block of tiles:

$$
\begin{bmatrix}
0 & 2 & \cdots\\
1 & 3 & \cdots
\end{bmatrix}
\quad(G=2)
\qquad\text{instead of row-major}\qquad
\begin{bmatrix}
0 & 1 & \cdots\\
G_N & G_N+1 & \cdots
\end{bmatrix}.
\tag{3.3}
$$

**Notes example.** $A_{6\times3}B_{3\times8}$, $B_M=3$, $B_N=4$, $G=2$:
$G_M=G_N=2$, $\mathrm{num\_pid\_in\_group}=4$. Tile order:

$$
\begin{bmatrix}0&2\\1&3\end{bmatrix}
\quad\Rightarrow\quad
\begin{aligned}
\mathrm{pid}\,0&\to C[0:3,0:4],\\
\mathrm{pid}\,1&\to C[3:6,0:4]\quad\text{(same $B$ column panel)},\\
\mathrm{pid}\,2&\to C[0:3,4:8],\\
\mathrm{pid}\,3&\to C[3:6,4:8].
\end{aligned}
\tag{3.4}
$$

### 3.1 Correctness: the swizzle only permutes tiles

**Theorem 4.** The map $\mathrm{pid}\mapsto(p_m,p_n)$ defined by (3.2) is a
bijection onto $[G_M]\times[G_N]$. Consequently the grouped kernel computes the
same $C$ as the blocked kernel.

*Proof.* The tile grid is partitioned into groups of $G$ consecutive tile rows
(possibly fewer in the last group), so group $g$ contains rows
$\{gG,\dots,gG+g_m-1\}$ and all $G_N$ columns, i.e. exactly $g_mG_N$ tiles. The
index $w\in[g_mG_N]$ is mapped to
$p_m=\text{first}+(w\bmod g_m)$, $p_n=\lfloor w/g_m\rfloor$; this is the standard
bijection $[g_m]\times[G_N]\to[g_mG_N]$ read in column-major order. Groups are
disjoint and cover all rows because $g_m=\min(G,G_M-\text{first})$. $\square$

**Corollary 5.** Grouped execution is independent of $G$; $G$ affects only
performance (which tiles are co-resident in L2), never the result.

### 3.2 Why column-major inside a group helps

Let the group contain $g_m$ tile rows and all $G_N$ tile columns. Processing in
column-major order means: fix a column panel $B[:,p_nB_N:(p_n+1)B_N]$, then sweep
$g_m$ different $A$ row panels against it.

- The $B$ panel is loaded once and reused $g_m$ times while it stays in L2.
- The $A$ panels of the group (there are $g_m$ of them) stay resident across the
  $G_N$ columns.

Balancing the two streams gives, for a square group $G\approx\sqrt{G_MG_N}$,

$$
\text{A streams}\approx G_M,
\qquad
\text{B streams}\approx \frac{G_M}{G}\,G_N,
\tag{3.5}
$$

i.e. the reload counts become comparable instead of one being $G_N\times$ the
other. The notes' illustrative count for a $3\times3$ tile grid with group size
$3$ is reproduced below; the exact constants depend on the cache model, but the
symmetry is the point:

| Order | A-panel loads | B-panel loads |
|-------|---------------|---------------|
| Row-major | $3$ | $9$ |
| Grouped | $3$ | $3$ |

**Corollary 6 (roofline).** If the grouped schedule raises L2 hit rate, the
effective DRAM traffic drops while FLOPs stay $2MNK$; by the roofline bound this
moves the kernel up the memory-bound slope and can expose the compute roof.

**Partial last group.** When $G\nmid G_M$, $g_m<G$ and the last group simply has
fewer rows. The formula (3.2) remains a bijection (Theorem 4), so no special
case is needed in the algebra.

### 3.3 Choosing $G$

- $G$ too small → behaves like row-major; $B$ reloads dominate.
- $G$ too large → the working set $G\cdot B_M\cdot K$ (for A) plus
  $K\cdot B_N\cdot G_N$ (for B) exceeds L2, so reuse is lost.
- Rule of thumb: choose $G$ so the group working set fits in L2, often
  $G\in\{4,8\}$ for the autotuned configurations in this notebook.


In [ ]:
def grouped_blocked_matmul_psuedo(A, B, BLOCK_SIZE_M, BLOCK_SIZE_N, GROUP_SIZE):
    """
    Blocking our matmul is a step in the right direction as it now enables proper
    (See markdown cells above for the math.)
    """

    M, K = A.shape
    K, N = B.shape
    C = torch.zeros(M, N, device=A.device, dtype=A.dtype)

    assert M % BLOCK_SIZE_M == 0, "Assuming M divisible by BLOCK_SIZE_M for simplicity"
    assert N % BLOCK_SIZE_N == 0, "Assuming N divisible by BLOCK_SIZE_N for simplicity"
    assert (BLOCK_SIZE_M % GROUP_SIZE == 0) and (
        BLOCK_SIZE_N % GROUP_SIZE == 0
    ), "Assuming BLOCK_SIZE_M/N are divisible by our GROUP_SIZE"

    ### Get how many rows and cols we have at the Block level ###
    num_block_rows = M // BLOCK_SIZE_M
    num_block_cols = N // BLOCK_SIZE_N

    num_pids = list(range(num_block_rows * num_block_cols))

    ### To visualize our groups we can just store the PID index in a matrix to print
    pid_visual = torch.zeros(num_block_rows, num_block_cols)
    for i in num_pids:
        ### Get number of Programs in Group ###
        ### This is for all columns, as in our example ###
        ### col_id:    0  1     2  3
        ### row_id: 0 [0  2  |  4  6 ]
        ### row_id: 1 [1  3  |  5  7 ]
        ###           ----------------
        ### row_id: 2 [8  10 | 12 14 ]
        ### row_id: 3 [9  11 | 13 15 ]

        ### to get to 8 (the start of the next group down the column)
        ### We need to get through 2 * 4 blocks. (group_size * num_block_cols)
        num_pid_in_group = GROUP_SIZE * num_block_cols

        ### Get Which Group We Are In (down the column) ###
        group_id = i // num_pid_in_group

        ### Now we can get the Row ID of the first program in the group
        ### This is the top of this specific group, we need to index further
        ### soon to get the exact row we are processing inside the group
        first_pid_m = group_id * GROUP_SIZE

        ### As a sanity check our group_size may not perfectly divide our num blocks
        ### and the last group may be smaller. In this psuedocode we dont worry about
        ### this as we assume divisible, but the triton code will have this so i put it here!
        group_size_m = min(num_block_rows - first_pid_m, GROUP_SIZE)

        ### Now that we have our row we are working on we need the column. And as we saw
        ### we do column major order within groups. (this part is trickier)
        where_in_group = i % num_pid_in_group

        ### Now we need to get ourselves into the correct row, as its column major.
        which_row = where_in_group % group_size_m

        ### And this is the row within the group, we just have to shift it over by where ever our starting group is
        pid_m = first_pid_m + which_row

        ### Now we get the column that this pid is supposed to compute.
        where_in_group = i % num_pid_in_group
        pid_n = where_in_group // group_size_m
        pid_visual[pid_m, pid_n] = i

        ### But we need to convert block_m, block_n to actual indexes in our data
        m = pid_m * BLOCK_SIZE_M
        n = pid_n * BLOCK_SIZE_N

        ### Grab the corresponding block of rows and cols from A and B
        block_A = A[m : m + BLOCK_SIZE_M, :]
        block_B = B[:, n : n + BLOCK_SIZE_N]

        ### Same matmul op on our block of data ###
        C[m : m + BLOCK_SIZE_M, n : n + BLOCK_SIZE_N] = naive_matmul_psuedo(block_A, block_B)

    return C, pid_visual


#### Triton blocked and grouped kernels

| Kernel | Program owns | Indexing |
|---|---|---|
| `naive_matmul_kernel` | one $C_{mn}$ | $m=\lfloor\mathrm{pid}/N\rfloor$, $n=\mathrm{pid}\bmod N$ |
| `blocked_matmul_kernel` | one $B_M\times B_N$ tile | row-major over tiles + $K$-panels of width $B_K$ |
| `grouped_matmul_kernel` | one tile | grouped swizzle of §3 + **same** $K$-loop / `tl.dot` / store |

**Accumulation.** Prefer an FP32 accumulator $\mathrm{Acc}$, then cast:

$$
C_{\mathrm{tile}}
=
\operatorname{cast}\!\left(\sum_{t=0}^{G_K-1}A_tB_t\right).
\tag{4.1}
$$

Masked loads implement the indicator $\mathbf{1}[\text{index in bounds}]$, so
shapes need not divide block sizes.

### 4.1 Pointer form

With offsets $o_m,o_n,o_k$:

$$
\begin{aligned}
A_{\mathrm{ptrs}}
&=a+o_m[:,\mathtt{None}]\,s_{am}+o_k[\mathtt{None},:]\,s_{ak},\\
B_{\mathrm{ptrs}}
&=b+o_k[:,\mathtt{None}]\,s_{bk}+o_n[\mathtt{None},:]\,s_{bn}.
\end{aligned}
\tag{4.2}
$$

Each $K$-step advances the pointers by

$$
A_{\mathrm{ptrs}}\leftarrow A_{\mathrm{ptrs}}+B_Ks_{ak},
\qquad
B_{\mathrm{ptrs}}\leftarrow B_{\mathrm{ptrs}}+B_Ks_{bk}.
\tag{4.3}
$$

Only the $\mathrm{pid}\mapsto(p_m,p_n)$ block differs between
`blocked_matmul_kernel` and `grouped_matmul_kernel`; the tile body is identical.

**Mask algebra.** For the last $K$-panel, let
$\kappa=K-kB_K$ be the number of remaining columns. Then

$$
A^{(k)}_{ij}
=
\begin{cases}
A_{o_m(i),\,kB_K+j}, & o_m(i)<M\ \wedge\ j<\kappa,\\
0, & \text{otherwise},
\end{cases}
\tag{4.4}
$$

and similarly for $B$. Loading zeros is the neutral element for the sum, so the
partial panel contributes exactly the in-range terms. This is why masking does
not change the result.

### 4.2 Numerical error

Let $u$ be the unit roundoff of the accumulator ($u=2^{-24}$ for FP32). The
computed tile satisfies the standard dot-product bound

$$
\big|\hat C_{mn}-C_{mn}\big|
\le
\gamma_{K}\,u\sum_{k=0}^{K-1}|A_{mk}||B_{kn}|,
\qquad
\gamma_K=\frac{Ku}{1-Ku}.
\tag{4.5}
$$

Equivalently, the relative error is $O(Ku)$ in the worst case and
$O(\sqrt{K}u)$ for random signs. With FP16 inputs but FP32 accumulation, the
dominant additional error is the input quantization, not the summation. This is
why the tests use `atol=rtol=1e-2` for FP16 and still match closely.

If the accumulator were FP16, $u=2^{-11}$ and the bound would be $\sim5000\times$
larger — the classic "FP16 accumulation ruins matmul" failure.

### 4.3 Autotuning

The `@triton.autotune` decorator sweeps configurations over
$(B_M,B_N,B_K,\text{warps},\text{stages})$ and keys on $(M,N,K)$. Each
configuration trades off:

- **tile size** → arithmetic intensity (0.6) and register/SMEM pressure;
- **num_warps** → parallelism per program;
- **num_stages** → software pipelining depth (overlap of loads and `tl.dot`).

The best point is where the tile fits in registers/SMEM and the K-loop is deep
enough to hide memory latency. That is a hardware-dependent optimum, hence the
autotuner rather than a hard-coded choice.

### 4.4 From arithmetic to roofline

Combining (0.3) and (0.6), the achieved performance obeys

$$
\frac{F}{t}
\le
\min\!\left(P_{\mathrm{peak}},\ \mathrm{BW}\cdot\mathrm{AI}_{\mathrm{tile}}\right).
\tag{4.6}
$$

For small tiles the second term binds (memory-bound); for large tiles the first
binds (compute-bound). Blocking moves the operating point from the memory roof
to the compute roof, and grouped ordering keeps it there by preserving L2 reuse.


In [ ]:
def get_blocked_cuda_autotune_config():
    return [
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 64}, num_stages=3, num_warps=8),
        triton.Config({"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 64, "BLOCK_SIZE_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 32, "BLOCK_SIZE_K": 32}, num_stages=4, num_warps=4),
        triton.Config({"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 32, "BLOCK_SIZE_K": 32}, num_stages=5, num_warps=2),
        triton.Config({"BLOCK_SIZE_M": 32, "BLOCK_SIZE_N": 64, "BLOCK_SIZE_K": 32}, num_stages=5, num_warps=2),
    ]


@triton.autotune(configs=get_blocked_cuda_autotune_config(), key=["M", "N", "K"])
@triton.jit
def blocked_matmul_kernel(
    A_ptr,
    B_ptr,
    C_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
):
    pid = tl.program_id(axis=0)

    ### Get how many columns of blocks we have ###
    num_block_cols = tl.cdiv(N, BLOCK_SIZE_N)

    ### Recover our actual block index ###
    block_m = pid // num_block_cols  # What row are we on?
    block_n = pid % num_block_cols  # What col are we on?

    ### Compute Offsets to grab full block of data ###
    offs_m = block_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_n = block_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    offs_k = tl.arange(0, BLOCK_SIZE_K)

    a_ptrs = A_ptr + (offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = B_ptr + (offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

    ### Initializer Accumulator - use float32 for accumulation ###
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    ### Loop over K in chunks of BLOCK_SIZE_K ###
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        ### Load Blocks of A and B with proper masking ###
        # Mask for A: check both M and K dimensions
        a_mask = (offs_m[:, None] < M) & (offs_k[None, :] < K - k * BLOCK_SIZE_K)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)

        # Mask for B: check both K and N dimensions
        b_mask = (offs_k[:, None] < K - k * BLOCK_SIZE_K) & (offs_n[None, :] < N)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)

        ### Accumulate Result ###
        ### investigate later: There is a discrepancy between pytorch and triton in float32
        ### if we dont include allow_fp32=False we wil have a higher error, but also,
        ### we then hurt performance on float32. Leaving as is for now!
        ###
        ### This is a well documented issue
        ### https://github.com/triton-lang/triton/issues/4574
        ### https://github.com/triton-lang/triton/issues/5204
        ### https://github.com/triton-lang/triton/issues/2843

        accumulator += tl.dot(a, b)  # allow_tf32=False

        ### Advance our Offsets to the next chunk of K ###
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk

    ### Cast accumulator to correct dtype ###
    c = accumulator.to(A_ptr.dtype.element_ty)

    ### Identify which block in our output C we will save this in ###
    c_offsets = stride_cm * offs_m[:, None] + stride_cn * offs_n[None, :]

    ### Identify any invalid positions we dont want to save in ###
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)

    ### Save it! ###
    tl.store(C_ptr + c_offsets, c, mask=c_mask)


def blocked_matmul(a, b):
    # Get dimensions
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, "Inner dimensions must match"

    # Initialize output tensor with same dtype as input
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)

    # Define grid
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_SIZE_M"]) * triton.cdiv(N, meta["BLOCK_SIZE_N"]),)

    # Launch kernel
    blocked_matmul_kernel[grid](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
    )
    return c


In [ ]:
def get_cuda_autotune_config():
    return [
        triton.Config(
            {"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 64, "GROUP_SIZE_M": 8},
            num_stages=3,
            num_warps=8,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 256, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=4,
            num_warps=4,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=4,
            num_warps=4,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 64, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=4,
            num_warps=4,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 128, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=4,
            num_warps=4,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 128, "BLOCK_SIZE_N": 32, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=4,
            num_warps=4,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 64, "BLOCK_SIZE_N": 32, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=5,
            num_warps=2,
        ),
        triton.Config(
            {"BLOCK_SIZE_M": 32, "BLOCK_SIZE_N": 64, "BLOCK_SIZE_K": 32, "GROUP_SIZE_M": 8},
            num_stages=5,
            num_warps=2,
        ),
    ]


@triton.autotune(configs=get_cuda_autotune_config(), key=["M", "N", "K"])
@triton.jit
def grouped_matmul_kernel(
    A_ptr,
    B_ptr,
    C_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_SIZE_M: tl.constexpr,
    BLOCK_SIZE_N: tl.constexpr,
    BLOCK_SIZE_K: tl.constexpr,
    GROUP_SIZE_M: tl.constexpr,
    DTYPE_FLAG: tl.constexpr,  # 0 for float32, 1 for float16
):
    ### Grouping logic as described above! ###
    pid = tl.program_id(axis=0)

    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n
    group_id = pid // num_pid_in_group
    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    ### EVERYTHING ELSE IS THE SAME AS BEFORE! ###
    ### Compute Offsets to grab full block of data ###
    offs_m = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    offs_k = tl.arange(0, BLOCK_SIZE_K)

    a_ptrs = A_ptr + (offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = B_ptr + (offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

    ### Initializer Accumulator - use float32 for accumulation ###
    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    ### Loop over K in chunks of BLOCK_SIZE_K ###
    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        ### Load Blocks of A and B with proper masking ###
        # Mask for A: check both M and K dimensions
        a_mask = (offs_m[:, None] < M) & (offs_k[None, :] < K - k * BLOCK_SIZE_K)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)

        # Mask for B: check both K and N dimensions
        b_mask = (offs_k[:, None] < K - k * BLOCK_SIZE_K) & (offs_n[None, :] < N)
        b = tl.load(b_ptrs, mask=b_mask, other=0.0)

        ### Accumulate Result ###
        accumulator += tl.dot(a, b)

        ### Advance our Offsets to the next chunk of K ###
        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk

    ### Cast accumulator to correct dtype ###
    c = accumulator.to(A_ptr.dtype.element_ty)

    ### Identify which block in our output C we will save this in ###
    c_offsets = stride_cm * offs_m[:, None] + stride_cn * offs_n[None, :]

    ### Identify any invalid positions we dont want to save in ###
    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)

    ### Save it! ###
    tl.store(C_ptr + c_offsets, c, mask=c_mask)


def fused_grouped_matmul(a, b):
    # Get dimensions
    M, K = a.shape
    K2, N = b.shape
    assert K == K2, "Inner dimensions must match"
    assert (a.dtype == torch.float32) or (a.dtype == torch.float16), "Only support float32 or float16 ops"
    assert a.dtype == b.dtype, f"a and b have different precision types {a.dtype} and {b.dtype}"

    # Initialize output tensor with same dtype as input
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)

    # Define grid
    grid = lambda meta: (triton.cdiv(M, meta["BLOCK_SIZE_M"]) * triton.cdiv(N, meta["BLOCK_SIZE_N"]),)

    # Launch kernel
    grouped_matmul_kernel[grid](
        a,
        b,
        c,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        c.stride(0),
        c.stride(1),
        DTYPE_FLAG=0 if a.dtype == torch.float32 else 1,
    )

    return c


# Alias used by later tutorials (08/09/10)
def matmul(a, b, activation=""):
    if activation not in ("", None):
        raise NotImplementedError(
            "this teaching matmul implements the plain GEMM; fuse the activation "
            "into the epilogue to support one"
        )
    return fused_grouped_matmul(a, b)


### Pitfalls

- Wrong stride order when building `a_ptrs`/`b_ptrs`.
- Accumulating FP16 without an FP32 accumulator → large error.
- Confusing tutorial-03 `GROUP_SIZE_M` (L2 swizzle) with tutorial-08 Group GEMM.
- Forgetting the $K$-remainder mask on the last panel.

### Self-check

1. Derive (0.3) and (0.6); evaluate both for the teaching sizes.
2. Hand-map $\mathrm{pid}\to(p_m,p_n)$ for $B_M=3$, $B_N=4$, $M=6$, $N=8$.
3. Write two $K$-panel steps for $B_K=4$, $K=8$ and state the loop invariant.
4. Why does grouped order reduce $B$-panel reloads?


In [ ]:
@pytest.mark.parametrize(
    "M, N, K",
    [
        (64, 64, 64),
        (128, 128, 128),
        (256, 256, 256),
        (512, 512, 512),
        (256, 512, 128),
        (512, 256, 128),
    ],
)
def test_matmul(M, N, K):
    require_gpu()
    # Create random inputs
    a = torch.randn(size=(M, K), dtype=torch.float16, device=DEVICE)
    b = torch.randn(size=(K, N), dtype=torch.float16, device=DEVICE)

    out_ref = torch.matmul(a, b)

    # Your fused kernel
    out = fused_grouped_matmul(a, b)

    # Validate results
    assert torch.allclose(out, out_ref, atol=1e-2, rtol=1e-2)

    # Log useful info
    max_diff = float(torch.max(torch.abs(out - out_ref)))
    print(f"Matmul Success: M={M}, N={N}, K={K} | Max Diff={max_diff:.4f}")


def test_correctness() -> None:
    test_matmul(64, 64, 64)


def test_pseudo_matches_torch() -> None:
    torch.manual_seed(0)
    a = torch.randn(6, 3)
    b = torch.randn(3, 8)
    ref = a @ b
    assert torch.allclose(naive_matmul_psuedo(a, b), ref, atol=1e-5)
    assert torch.allclose(blocked_matmul_pseudo(a, b, 3, 4), ref, atol=1e-5)
    grouped, pid_visual = grouped_blocked_matmul_psuedo(a, b, 3, 4, GROUP_SIZE=2)
    assert torch.allclose(grouped, ref, atol=1e-5)
    # GROUP_SIZE=2 on a 2x2 tile grid must produce column-major order in each
    # group: [[0, 2], [1, 3]], not the row-major [[0, 1], [2, 3]].
    assert pid_visual.tolist() == [[0.0, 2.0], [1.0, 3.0]]


In [ ]:
configs = [
    triton.testing.Benchmark(
        x_names=["M", "N", "K"],
        x_vals=[512 * i for i in range(2, 9)],
        line_arg="provider",
        line_vals=["torch", "blocked_matmul", "grouped_matmul"],
        line_names=["PyTorch", "Matmul No Groups", "Matmul Grouped"],
        styles=[("blue", "-"), ("purple", "-"), ("green", "-")],
        ylabel="TFLOPS",
        plot_name="matmul",
        args={},
    )
]


@triton.testing.perf_report(configs)
def benchmark_matmul(M, N, K, provider):
    if provider == "torch":
        a = torch.randn((M, K), device=DEVICE, dtype=torch.float16)
        b = torch.randn((K, N), device=DEVICE, dtype=torch.float16)
        ms = triton.testing.do_bench(lambda: torch.matmul(a, b))
    if provider == "blocked_matmul":
        a = torch.randn((M, K), device=DEVICE, dtype=torch.float16)
        b = torch.randn((K, N), device=DEVICE, dtype=torch.float16)
        ms = triton.testing.do_bench(lambda: blocked_matmul(a, b))
    if provider == "grouped_matmul":
        a = torch.randn((M, K), device=DEVICE, dtype=torch.float16)
        b = torch.randn((K, N), device=DEVICE, dtype=torch.float16)
        ms = triton.testing.do_bench(lambda: fused_grouped_matmul(a, b))

    perf = lambda ms: 2 * M * N * K * 1e-12 / (ms * 1e-3)
    return perf(ms)


In [ ]:
def test_tail_shapes() -> None:
    """GPU: M, N, K not divisible by the tile sizes exercise every mask path."""
    require_gpu()
    torch.manual_seed(0)
    m, n, k = 100, 100, 100
    a = torch.randn(m, k, device=DEVICE, dtype=torch.float16)
    b = torch.randn(k, n, device=DEVICE, dtype=torch.float16)
    ref = a @ b
    for name, fn in (
        ("naive", naive_matmul),
        ("blocked", blocked_matmul),
        ("grouped", fused_grouped_matmul),
    ):
        out = fn(a, b)
        assert torch.allclose(out.float(), ref.float(), atol=2e-2, rtol=2e-2), name
        print(f"{name} tail shapes OK")
